[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/02_operators.ipynb)

*Python from Scratch · Notebook 2 of 12*

# Operators & Expressions

Arithmetic, comparison, logical, assignment, identity, membership and bitwise operators — and the order Python applies them.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Operators are the verbs of a program: they take values and produce a new one. Order of operations works like school maths, with a few extras.

**What interviewers really test here:** // and % with negatives, `and`/`or` returning values, `is` vs `==`, and float comparison.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Arithmetic operators *(Basics)*
- [ ] 2. Comparison and logical operators *(Basics)*
- [ ] 3. Assignment operators *(Core)*
- [ ] 4. Order of operations (precedence) *(Core)*
- [ ] 5. Short-circuit evaluation, identity and membership *(Advanced)*
- [ ] 6. Bitwise operators *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definitions

An **operator** is a symbol (or keyword) that performs an operation on values called **operands**. In `3 + 4`, `+` is the operator and 3 and 4 are operands.



An **expression** is any combination of values, variables, operators and function calls that Python can **evaluate** to produce a single value: `price * qty + tax`. A **statement** is a complete instruction, such as an assignment `total = price * qty`.



Operators are **unary** (one operand: `-x`, `not x`) or **binary** (two operands: `a + b`). Python also has a **ternary** conditional expression: `a if cond else b`.

### Categories of operators

| Category | Operators | Result |
| --- | --- | --- |
| Arithmetic | `+ - * / // % **` | number |
| Comparison (relational) | `== != < > <= >=` | bool |
| Logical | `and or not` | one of the operands (truthy/falsy) |
| Assignment | `= += -= *= /= //= %= **= &= \|= ^= <<= >>=`, walrus `:=` | — (statement) |
| Identity | `is`, `is not` | bool |
| Membership | `in`, `not in` | bool |
| Bitwise | `& \| ^ ~ << >>` | int |



The **walrus operator** `:=` (Python 3.8+) assigns inside an expression: `if (n := len(data)) > 10:`.

### Division in detail

| Expression | Result | Rule |
| --- | --- | --- |
| `7 / 2` | 3.5 | **true division**, always float |
| `6 / 2` | 3.0 | still a float |
| `7 // 2` | 3 | **floor division**: rounds toward −∞ |
| `-7 // 2` | −4 | floor, not truncation |
| `7 % 2` | 1 | remainder |
| `-7 % 2` | 1 | result has the sign of the divisor |
| `7.5 // 2` | 3.0 | float in → float out |
| `1 / 0` | ZeroDivisionError | also for // and % |



Identity that always holds: `a == (a // b) * b + (a % b)`. `divmod(a, b)` returns `(a // b, a % b)`.

### Precedence and associativity

| Priority | Operators | Associativity |
| --- | --- | --- |
| 1 (highest) | `( )` | — |
| 2 | `**` | right to left |
| 3 | `+x -x ~x` | right to left |
| 4 | `* / // %` | left to right |
| 5 | `+ -` | left to right |
| 6 | `<< >>` | left to right |
| 7 | `&` | left to right |
| 8 | `^` | left to right |
| 9 | `\|` | left to right |
| 10 | comparisons, `is`, `in` | chained |
| 11 | `not` | — |
| 12 | `and` | left to right |
| 13 | `or` | left to right |
| 14 (lowest) | `if … else`, `lambda`, `:=` | — |



**Associativity** decides the order among operators of equal priority: `10 - 4 - 3` = (10 − 4) − 3 = 3, but `2 ** 3 ** 2` = 2 \*\* (3 \*\* 2) = 512.

### Truthiness and short-circuit evaluation

Any value can be used where a bool is expected. **Falsy** values: `False`, `None`, `0`, `0.0`, `""`, `[]`, `()`, `{}`, `set()`, `range(0)`. Everything else is **truthy**.




- `x and y`: if x is falsy, returns x (y is never evaluated); otherwise returns y.
- `x or y`: if x is truthy, returns x (y is never evaluated); otherwise returns y.
- `not x`: always returns a bool.



Short-circuiting lets you guard expressions: `if user and user.is_admin` never touches `user.is_admin` when user is None.

### Operators on other types

| Expression | Result | Meaning |
| --- | --- | --- |
| `"ab" + "cd"` | "abcd" | concatenation |
| `"ab" * 3` | "ababab" | repetition |
| `[1, 2] + [3]` | [1, 2, 3] | list concatenation |
| `"a" < "b"` | True | strings compare alphabetically (by Unicode code) |
| `[1, 2] == [1, 2]` | True | element-wise equality |
| `{1, 2} \| {2, 3}` | {1, 2, 3} | set union |
| `"5" + 3` | TypeError | no implicit str/int mixing |



This works because operators call **special methods**: `a + b` calls `a.__add__(b)`. Your own classes can define them (**operator overloading**, see the OOP topic).

---
# Part 2: Lessons, step by step

## Lesson 1: Arithmetic operators

*Level: Basics*

| Operator | Name | Example | Result |
| --- | --- | --- | --- |
| `+` | add | `7 + 2` | 9 |
| `-` | subtract | `7 - 2` | 5 |
| `*` | multiply | `7 * 2` | 14 |
| `/` | divide (always gives a float) | `7 / 2` | 3.5 |
| `//` | floor division (round down) | `7 // 2` | 3 |
| `%` | modulo (remainder) | `7 % 2` | 1 |
| `**` | power | `7 ** 2` | 49 |



`//` and `%` are a pair: `a == (a // b) * b + a % b`. They answer "how many whole times does b fit?" and "what's left over?".



Uses of `%`: even/odd (`n % 2`), last digit (`n % 10`), wrapping around (`(i + 1) % 7` for days of the week).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def arithmetic_demo():
    a = 17
    b = 5
    print(a + b, a - b, a * b)
    print(a / b, a // b, a % b)
    print(2 ** 10)
    check = (a // b) * b + a % b
    print(check == a)


arithmetic_demo()

22 12 85
3.4 3 2
1024
True


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q10](#q10), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 2: Comparison and logical operators

*Level: Basics*

Comparisons give a **bool** (True or False): `==` equal, `!=` not equal, `<`, `>`, `<=`, `>=`.



Logical operators combine bools:



| a | b | a and b | a or b | not a |
| --- | --- | --- | --- | --- |
| True | True | True | True | False |
| True | False | False | True | False |
| False | True | False | True | True |
| False | False | False | False | True |



Python lets you **chain** comparisons like maths: `0 <= score <= 100`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def logic_demo():
    age = 19
    has_ticket = True
    is_adult = age >= 18
    can_enter = is_adult and has_ticket
    is_teen = 13 <= age <= 19
    print(is_adult, can_enter, is_teen, not can_enter)


logic_demo()

True True True False


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q13](#q13), [Q12](#q12), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 3: Assignment operators

*Level: Core*

Updating a variable with itself is so common that Python has shortcuts:



| Shortcut | Same as |
| --- | --- |
| `x += 3` | `x = x + 3` |
| `x -= 3` | `x = x - 3` |
| `x *= 3` | `x = x * 3` |
| `x /= 3` | `x = x / 3` |
| `x //= 3`, `x %= 3`, `x **= 3` | same pattern |



They work on strings and lists too: `s += "!"`. Python has **no** `x++` — write `x += 1`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def assign_demo():
    balance = 100
    balance += 50
    balance -= 30
    balance *= 2
    balance //= 3
    message = "Balance"
    message += ": " + str(balance)
    print(message)


assign_demo()

Balance: 80


**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 4: Order of operations (precedence)

*Level: Core*

Like in maths, some operators go first. From highest to lowest:




1. `( )` brackets
2. `**` power (right to left: `2 ** 3 ** 2` = 2 \*\* 9 = 512)
3. unary `-x`, `+x`
4. `* / // %`
5. `+ -`
6. comparisons `== != < > <= >= is in`
7. `not`
8. `and`
9. `or`



Operators on the same level run left to right. `-2 ** 2` is `-4` because `**` beats unary minus.



**💡 Tip:** Brackets cost nothing and make intent obvious. Use them whenever an expression mixes kinds of operators.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def precedence_demo():
    a = 2 + 3 * 4
    b = (2 + 3) * 4
    c = 2 ** 3 ** 2
    d = -2 ** 2
    e = 10 - 4 - 3
    f = True or False and False
    print(a, b, c, d, e, f)


precedence_demo()

14 20 512 -4 3 True


**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 5: Short-circuit evaluation, identity and membership

*Level: Advanced*

#### Short-circuit



`and` stops at the first falsy value; `or` stops at the first truthy value — the rest isn't even evaluated. They also return one of the **operands**, not just True/False:



```python
name = "" or "Guest"          # "Guest"
x = 0 and 10/0                # 0  (10/0 never runs)
```



#### Identity and membership




- `a is b` — the same object? Use it for `None`: `if x is None`.
- `x in collection` — is x inside a string, list, tuple, set or dict keys?

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def short_circuit_demo():
    name = ""
    shown = name or "Guest"
    count = 0
    safe = count != 0 and 100 / count > 5
    vowels = "aeiou"
    has_e = "e" in vowels
    x = None
    is_missing = x is None
    print(shown, safe, has_e, is_missing)


short_circuit_demo()

Guest False True True


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 6: Bitwise operators

*Level: Advanced*

Bitwise operators work on the binary digits of integers:



| Op | Meaning | `6` (110) with `3` (011) |
| --- | --- | --- |
| `&` | AND each bit | 010 = 2 |
| `\|` | OR each bit | 111 = 7 |
| `^` | XOR each bit | 101 = 5 |
| `~` | flip all bits | `~6` = −7 |
| `<<`, `>>` | shift left / right | `6 << 1` = 12, `6 >> 1` = 3 |



They're used for flags and permissions, fast checks like "power of two" (`n & (n - 1) == 0`), and many puzzle questions. The DSA handbook has a full topic on them.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def bitwise_demo():
    a = 6
    b = 3
    print(bin(a), bin(b))
    print(a & b, a | b, a ^ b)
    print(a << 1, a >> 1, ~a)
    n = 8
    print(n & (n - 1) == 0)


bitwise_demo()

0b110 0b11
2 7 5
12 3 -7
True


**✏️ Your turn:** you now have what you need for [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Even or Odd

🟢 Warm-up · *% operator*

Return `'even'` if `n` is even, otherwise `'odd'`.

**Examples**

- `n = 4` → `'even'`
- `n = 7` → `'odd'`
- `n = 0` → `'even'`

In [ ]:
def even_or_odd(n):
    """
    Return 'even' if n is even, otherwise 'odd'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(even_or_odd, [
    ((4,), 'even'),
    ((7,), 'odd'),
    ((0,), 'even'),
    ((-3,), 'odd'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep subtracting 2 (or adding 2 for negatives) until you reach 0 or 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A number is even exactly when `n % 2 == 0`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Keep subtracting 2 (or adding 2 for negatives) until you reach 0 or 1.

```python
def even_or_odd(n):
    n = abs(n)
    while n >= 2:
        n = n - 2
    if n == 0:
        return "even"
    return "odd"
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A number is even exactly when `n % 2 == 0`.

```python
def even_or_odd(n):
    return "even" if n % 2 == 0 else "odd"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep subtracting 2 (or adding 2 for negatives) until you reach 0 or 1. That costs O(n) time and O(1) space."

"I can do better: A number is even exactly when `n % 2 == 0`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/even-or-odd)

<a id="q2"></a>
### Q2. Leap Year

🟢 Warm-up · *and / or*

A year is a leap year if it's divisible by 4, except years divisible by 100, unless also divisible by 400. Return True or False.

**Examples**

- `year = 2024` → `True`
- `year = 1900` → `False`
- `year = 2000` → `True`

In [ ]:
def is_leap(year):
    """
    A year is a leap year if it's divisible by 4, except years divisible by 100,
    unless also divisible by 400. Return True or False.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_leap, [
    ((2024,), True),
    ((1900,), False),
    ((2000,), True),
    ((2023,), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested `if` statements, one rule at a time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Combine the rules into one boolean expression with `and`/`or`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Nested `if` statements, one rule at a time.

```python
def is_leap(year):
    if year % 4 == 0:
        if year % 100 == 0:
            if year % 400 == 0:
                return True
            return False
        return True
    return False
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Combine the rules into one boolean expression with `and`/`or`.

```python
def is_leap(year):
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested `if` statements, one rule at a time. That costs O(1) time and O(1) space."

"I can do better: Combine the rules into one boolean expression with `and`/`or`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/leap-year)

<a id="q3"></a>
### Q3. Power Without \*\*

🟢 Warm-up · *\*\* · loops*

Return `base` raised to a non-negative integer `exp`.

**Examples**

- `base = 2, exp = 10` → `1024`
- `base = 5, exp = 0` → `1`
- `base = 3, exp = 4` → `81`

In [ ]:
def power(base, exp):
    """
    Return base raised to a non-negative integer exp.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(power, [
    ((2, 10), 1024),
    ((5, 0), 1),
    ((3, 4), 81),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply a result by base, exp times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Python's `**` operator (or `pow`) does it directly.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Multiply a result by base, exp times.

```python
def power(base, exp):
    result = 1
    for _ in range(exp):
        result = result * base
    return result
```

**Cost:** O(exp) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Python's `**` operator (or `pow`) does it directly.

```python
def power(base, exp):
    return base ** exp
```

**Cost:** O(log exp) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply a result by base, exp times. That costs O(exp) time and O(1) space."

"I can do better: Python's `**` operator (or `pow`) does it directly. That's O(log exp) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/power-without)

<a id="q4"></a>
### Q4. Absolute Value

🟢 Warm-up · *Unary minus · abs()*

Return the absolute value (distance from zero) of `n`.

**Examples**

- `n = -8` → `8`
- `n = 5` → `5`
- `n = 0` → `0`

In [ ]:
def absolute(n):
    """
    Return the absolute value (distance from zero) of n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(absolute, [
    ((-8,), 8),
    ((5,), 5),
    ((0,), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If n is negative, flip its sign with unary minus.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the built-in `abs()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** If n is negative, flip its sign with unary minus.

```python
def absolute(n):
    if n < 0:
        n = -n
    return n
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the built-in `abs()`.

```python
def absolute(n):
    return abs(n)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If n is negative, flip its sign with unary minus. That costs O(1) time and O(1) space."

"I can do better: Use the built-in `abs()`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/absolute-value)

<a id="q5"></a>
### Q5. Largest of Three

🟢 Warm-up · *Comparison operators*

Return the largest of `a`, `b`, `c`.

**Examples**

- `a = 3, b = 9, c = 4` → `9`
- `a = 7, b = 7, c = 2` → `7`
- `a = -1, b = -5, c = -3` → `-1`

In [ ]:
def largest(a, b, c):
    """
    Return the largest of a, b, c.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(largest, [
    ((3, 9, 4), 9),
    ((7, 7, 2), 7),
    ((-1, -5, -3), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare with `>=` using if/elif.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Built-in `max()` takes any number of values.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare with `>=` using if/elif.

```python
def largest(a, b, c):
    if a >= b and a >= c:
        return a
    elif b >= a and b >= c:
        return b
    else:
        return c
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Built-in `max()` takes any number of values.

```python
def largest(a, b, c):
    return max(a, b, c)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare with `>=` using if/elif. That costs O(1) time and O(1) space."

"I can do better: Built-in `max()` takes any number of values. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/largest-of-three)

<a id="q6"></a>
### Q6. FizzBuzz Word

🟢 Warm-up · *% · and*

Return `'FizzBuzz'` if n is divisible by 3 and 5, `'Fizz'` if only by 3, `'Buzz'` if only by 5, else the number as a string.

**Examples**

- `n = 15` → `'FizzBuzz'`
- `n = 9` → `'Fizz'`
- `n = 10` → `'Buzz'`

In [ ]:
def fizzbuzz(n):
    """
    Return 'FizzBuzz' if n is divisible by 3 and 5, 'Fizz' if only by 3, 'Buzz'
    if only by 5, else the number as a string.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fizzbuzz, [
    ((15,), 'FizzBuzz'),
    ((9,), 'Fizz'),
    ((10,), 'Buzz'),
    ((7,), '7'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check the combined case first, then each single case.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build the word by adding 'Fizz' and 'Buzz' pieces; fall back to the number with `or`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check the combined case first, then each single case.

```python
def fizzbuzz(n):
    if n % 3 == 0 and n % 5 == 0:
        return "FizzBuzz"
    elif n % 3 == 0:
        return "Fizz"
    elif n % 5 == 0:
        return "Buzz"
    else:
        return str(n)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Build the word by adding 'Fizz' and 'Buzz' pieces; fall back to the number with `or`.

```python
def fizzbuzz(n):
    word = ("Fizz" if n % 3 == 0 else "") + ("Buzz" if n % 5 == 0 else "")
    return word or str(n)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check the combined case first, then each single case. That costs O(1) time and O(1) space."

"I can do better: Build the word by adding 'Fizz' and 'Buzz' pieces; fall back to the number with `or`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/fizzbuzz-word)

<a id="q7"></a>
### Q7. Compound Interest

🟢 Warm-up · *\*\* operator*

Return the amount after `years` with yearly compound interest `rate` percent, rounded to 2 decimals.

**Examples**

- `principal = 1000, rate = 10, years = 2` → `1210.0`
- `principal = 500, rate = 5, years = 3` → `578.81`
- `principal = 100, rate = 0, years = 5` → `100.0`

In [ ]:
def compound(principal, rate, years):
    """
    Return the amount after years with yearly compound interest rate percent,
    rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compound, [
    ((1000, 10, 2), 1210.0),
    ((500, 5, 3), 578.81),
    ((100, 0, 5), 100.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Apply the interest year by year in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the formula P × (1 + r/100) \*\* years.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Apply the interest year by year in a loop.

```python
def compound(principal, rate, years):
    amount = principal
    for _ in range(years):
        amount = amount + amount * rate / 100
    return round(amount, 2)
```

**Cost:** O(years) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the formula P × (1 + r/100) \*\* years.

```python
def compound(principal, rate, years):
    return round(principal * (1 + rate / 100) ** years, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Apply the interest year by year in a loop. That costs O(years) time and O(1) space."

"I can do better: Use the formula P × (1 + r/100) \*\* years. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/compound-interest)

<a id="q8"></a>
### Q8. Perfect Square

🟢 Warm-up · *\*\* · math.isqrt*

Return True if `n` (≥ 0) is a perfect square like 16 or 25.

**Examples**

- `n = 16` → `True`
- `n = 15` → `False`
- `n = 0` → `True`

In [ ]:
def is_square(n):
    """
    Return True if n (≥ 0) is a perfect square like 16 or 25.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_square, [
    ((16,), True),
    ((15,), False),
    ((0,), True),
    ((1,), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try i = 0, 1, 2, … while i × i ≤ n.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.isqrt(n)` is the whole-number square root; square it and compare.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Try i = 0, 1, 2, … while i × i ≤ n.

```python
def is_square(n):
    i = 0
    while i * i <= n:
        if i * i == n:
            return True
        i += 1
    return False
```

**Cost:** O(√n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `math.isqrt(n)` is the whole-number square root; square it and compare.

```python
def is_square(n):
    import math
    r = math.isqrt(n)
    return r * r == n
```

**Cost:** O(1)\* time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try i = 0, 1, 2, … while i × i ≤ n. That costs O(√n) time and O(1) space."

"I can do better: `math.isqrt(n)` is the whole-number square root; square it and compare. That's O(1)\* time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/perfect-square)

<a id="q9"></a>
### Q9. Floats Almost Equal

🟢 Warm-up · *Float comparison*

Return True if `a` and `b` are equal up to tiny rounding errors (e.g. 0.1 + 0.2 and 0.3).

**Examples**

- `a = 0.30000000000000004, b = 0.3` → `True`
- `a = 1.0, b = 1.001` → `False`
- `a = 2.5, b = 2.5` → `True`

In [ ]:
def almost_equal(a, b):
    """
    Return True if a and b are equal up to tiny rounding errors (e.g. 0.1 + 0.2
    and 0.3).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(almost_equal, [
    ((0.30000000000000004, 0.3), True),
    ((1.0, 1.001), False),
    ((2.5, 2.5), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check that the difference is smaller than a tiny tolerance.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.isclose` uses a sensible relative tolerance for you.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check that the difference is smaller than a tiny tolerance.

```python
def almost_equal(a, b):
    difference = a - b
    if difference < 0:
        difference = -difference
    return difference < 1e-9
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `math.isclose` uses a sensible relative tolerance for you.

```python
def almost_equal(a, b):
    import math
    return math.isclose(a, b)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check that the difference is smaller than a tiny tolerance. That costs O(1) time and O(1) space."

"I can do better: `math.isclose` uses a sensible relative tolerance for you. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/floats-almost-equal)

<a id="q10"></a>
### Q10. Ceiling Division

🟢 Warm-up · *// trick*

How many boxes of size `b` are needed for `a` items? Return ⌈a / b⌉ for positive integers.

**Examples**

- `a = 10, b = 3` → `4`
- `a = 9, b = 3` → `3`
- `a = 1, b = 5` → `1`

In [ ]:
def ceil_div(a, b):
    """
    How many boxes of size b are needed for a items? Return ⌈a / b⌉ for positive
    integers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ceil_div, [
    ((10, 3), 4),
    ((9, 3), 3),
    ((1, 5), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide with //, then add 1 if there's a remainder.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Negating twice turns floor into ceiling: `-(-a // b)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Divide with //, then add 1 if there's a remainder.

```python
def ceil_div(a, b):
    boxes = a // b
    if a % b != 0:
        boxes += 1
    return boxes
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Negating twice turns floor into ceiling: `-(-a // b)`.

```python
def ceil_div(a, b):
    return -(-a // b)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide with //, then add 1 if there's a remainder. That costs O(1) time and O(1) space."

"I can do better: Negating twice turns floor into ceiling: `-(-a // b)`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/ceiling-division)

<a id="q11"></a>
### Q11. Count Digits

🟢 Warm-up · *// · len(str())*

Return how many digits the integer `n` has (ignore the sign; 0 has 1 digit).

**Examples**

- `n = 12345` → `5`
- `n = -70` → `2`
- `n = 0` → `1`

In [ ]:
def count_digits(n):
    """
    Return how many digits the integer n has (ignore the sign; 0 has 1 digit).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_digits, [
    ((12345,), 5),
    ((-70,), 2),
    ((0,), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide by 10 until it reaches 0, counting steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Length of the number written as text.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Divide by 10 until it reaches 0, counting steps.

```python
def count_digits(n):
    n = abs(n)
    count = 1
    while n >= 10:
        n //= 10
        count += 1
    return count
```

**Cost:** O(d) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Length of the number written as text.

```python
def count_digits(n):
    return len(str(abs(n)))
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide by 10 until it reaches 0, counting steps. That costs O(d) time and O(1) space."

"I can do better: Length of the number written as text. That's O(d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/count-digits)

<a id="q12"></a>
### Q12. Exactly One Is True

🟢 Warm-up · *Logical operators*

Given two booleans, return True if exactly one of them is True (exclusive or).

**Examples**

- `a = True, b = False` → `True`
- `a = True, b = True` → `False`
- `a = False, b = False` → `False`

In [ ]:
def exactly_one(a, b):
    """
    Given two booleans, return True if exactly one of them is True (exclusive
    or).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(exactly_one, [
    ((True, False), True),
    ((True, True), False),
    ((False, False), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Spell out both cases with `and`, `or` and `not`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

For booleans, 'exactly one' is the same as 'they are different': `a != b`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Spell out both cases with `and`, `or` and `not`.

```python
def exactly_one(a, b):
    return (a and not b) or (b and not a)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** For booleans, 'exactly one' is the same as 'they are different': `a != b`.

```python
def exactly_one(a, b):
    return a != b
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Spell out both cases with `and`, `or` and `not`. That costs O(1) time and O(1) space."

"I can do better: For booleans, 'exactly one' is the same as 'they are different': `a != b`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/exactly-one-is-true)

<a id="q13"></a>
### Q13. Within a Range

🟢 Warm-up · *Chained comparison*

Return True if `lo ≤ x ≤ hi`.

**Examples**

- `x = 5, lo = 1, hi = 10` → `True`
- `x = 0, lo = 1, hi = 10` → `False`
- `x = 10, lo = 1, hi = 10` → `True`

In [ ]:
def in_range(x, lo, hi):
    """
    Return True if lo ≤ x ≤ hi.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(in_range, [
    ((5, 1, 10), True),
    ((0, 1, 10), False),
    ((10, 1, 10), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two comparisons joined by `and`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Python allows chaining: `lo <= x <= hi`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Two comparisons joined by `and`.

```python
def in_range(x, lo, hi):
    return x >= lo and x <= hi
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Python allows chaining: `lo <= x <= hi`.

```python
def in_range(x, lo, hi):
    return lo <= x <= hi
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two comparisons joined by `and`. That costs O(1) time and O(1) space."

"I can do better: Python allows chaining: `lo <= x <= hi`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/within-a-range)

<a id="q14"></a>
### Q14. Sum From 1 to n

🟢 Warm-up · *Formula vs loop*

Return 1 + 2 + … + n.

**Examples**

- `n = 10` → `55`
- `n = 1` → `1`
- `n = 100` → `5050`

In [ ]:
def sum_to(n):
    """
    Return 1 + 2 + … + n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_to, [
    ((10,), 55),
    ((1,), 1),
    ((100,), 5050),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add every number in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Gauss's formula n × (n + 1) // 2.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Add every number in a loop.

```python
def sum_to(n):
    total = 0
    for i in range(1, n + 1):
        total += i
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Gauss's formula n × (n + 1) // 2.

```python
def sum_to(n):
    return n * (n + 1) // 2
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add every number in a loop. That costs O(n) time and O(1) space."

"I can do better: Gauss's formula n × (n + 1) // 2. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/sum-from-1-to-n)

<a id="q15"></a>
### Q15. Number of Handshakes

🟢 Warm-up · *Formula vs nested loop*

In a room of `n` people, everyone shakes hands with everyone else once. How many handshakes?

**Examples**

- `n = 4` → `6`
- `n = 1` → `0`
- `n = 10` → `45`

In [ ]:
def handshakes(n):
    """
    In a room of n people, everyone shakes hands with everyone else once. How
    many handshakes?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(handshakes, [
    ((4,), 6),
    ((1,), 0),
    ((10,), 45),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count every pair (i, j) with i &lt; j using two loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Each of n people shakes n − 1 hands, and each handshake is counted twice: n(n − 1) / 2.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count every pair (i, j) with i &lt; j using two loops.

```python
def handshakes(n):
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            count += 1
    return count
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Each of n people shakes n − 1 hands, and each handshake is counted twice: n(n − 1) / 2.

```python
def handshakes(n):
    return n * (n - 1) // 2
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count every pair (i, j) with i &lt; j using two loops. That costs O(n²) time and O(1) space."

"I can do better: Each of n people shakes n − 1 hands, and each handshake is counted twice: n(n − 1) / 2. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/number-of-handshakes)

<a id="q16"></a>
### Q16. Clamp a Value

🟢 Warm-up · *min and max*

Return `x` limited to the range [lo, hi]: below lo becomes lo, above hi becomes hi.

**Examples**

- `x = 15, lo = 0, hi = 10` → `10`
- `x = -3, lo = 0, hi = 10` → `0`
- `x = 7, lo = 0, hi = 10` → `7`

In [ ]:
def clamp(x, lo, hi):
    """
    Return x limited to the range [lo, hi]: below lo becomes lo, above hi
    becomes hi.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clamp, [
    ((15, 0, 10), 10),
    ((-3, 0, 10), 0),
    ((7, 0, 10), 7),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two if statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max(lo, min(x, hi))` in one line.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Two if statements.

```python
def clamp(x, lo, hi):
    if x < lo:
        return lo
    if x > hi:
        return hi
    return x
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `max(lo, min(x, hi))` in one line.

```python
def clamp(x, lo, hi):
    return max(lo, min(x, hi))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two if statements. That costs O(1) time and O(1) space."

"I can do better: `max(lo, min(x, hi))` in one line. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/clamp-a-value)

<a id="q17"></a>
### Q17. Reverse a Number

🟡 Interview standard · *// and %*

Return the digits of a positive integer in reverse order as an int, e.g. 1230 → 321.

**Examples**

- `n = 1234` → `4321`
- `n = 1230` → `321`
- `n = 7` → `7`

In [ ]:
def reverse_number(n):
    """
    Return the digits of a positive integer in reverse order as an int, e.g.
    1230 → 321.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_number, [
    ((1234,), 4321),
    ((1230,), 321),
    ((7,), 7),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Take the last digit with % 10, append it to the result with result × 10 + digit, drop it with // 10.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Turn it into a string, reverse with a slice, and convert back.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Take the last digit with % 10, append it to the result with result × 10 + digit, drop it with // 10.

```python
def reverse_number(n):
    result = 0
    while n > 0:
        digit = n % 10
        result = result * 10 + digit
        n //= 10
    return result
```

**Cost:** O(d) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Turn it into a string, reverse with a slice, and convert back.

```python
def reverse_number(n):
    return int(str(n)[::-1])
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Take the last digit with % 10, append it to the result with result × 10 + digit, drop it with // 10. That costs O(d) time and O(1) space."

"I can do better: Turn it into a string, reverse with a slice, and convert back. That's O(d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/reverse-a-number)

<a id="q18"></a>
### Q18. Power of Two

🟡 Interview standard · *Bitwise &*

Return True if `n` is a power of two (1, 2, 4, 8, …).

**Examples**

- `n = 8` → `True`
- `n = 12` → `False`
- `n = 1` → `True`

In [ ]:
def is_power_of_two(n):
    """
    Return True if n is a power of two (1, 2, 4, 8, …).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_power_of_two, [
    ((8,), True),
    ((12,), False),
    ((1,), True),
    ((0,), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide by 2 while even; it's a power of two if you end at 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A power of two has a single 1-bit, so `n & (n - 1)` is 0.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Divide by 2 while even; it's a power of two if you end at 1.

```python
def is_power_of_two(n):
    if n <= 0:
        return False
    while n % 2 == 0:
        n //= 2
    return n == 1
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A power of two has a single 1-bit, so `n & (n - 1)` is 0.

```python
def is_power_of_two(n):
    return n > 0 and n & (n - 1) == 0
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide by 2 while even; it's a power of two if you end at 1. That costs O(log n) time and O(1) space."

"I can do better: A power of two has a single 1-bit, so `n & (n - 1)` is 0. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/power-of-two)

<a id="q19"></a>
### Q19. Round to Nearest Multiple

🟡 Interview standard · *// and arithmetic*

Round a non-negative integer `n` to the nearest multiple of `k` (halves round up).

**Examples**

- `n = 23, k = 5` → `25`
- `n = 22, k = 5` → `20`
- `n = 15, k = 10` → `20`

In [ ]:
def round_to(n, k):
    """
    Round a non-negative integer n to the nearest multiple of k (halves round
    up).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(round_to, [
    ((23, 5), 25),
    ((22, 5), 20),
    ((15, 10), 20),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Look at the multiples just below and just above n, and pick the closer one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Add half of k and floor-divide: `(n + k // 2) // k * k`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Look at the multiples just below and just above n, and pick the closer one.

```python
def round_to(n, k):
    below = n - n % k
    above = below + k
    if n - below < above - n:
        return below
    return above
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Add half of k and floor-divide: `(n + k // 2) // k * k`.

```python
def round_to(n, k):
    return (n + k // 2) // k * k
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Look at the multiples just below and just above n, and pick the closer one. That costs O(1) time and O(1) space."

"I can do better: Add half of k and floor-divide: `(n + k // 2) // k * k`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/round-to-nearest-multiple)

<a id="q20"></a>
### Q20. Weekly Pay With Overtime

🟡 Interview standard · *min / max arithmetic*

Pay is `rate` per hour for the first 40 hours and 1.5 × rate for every hour above 40. Return total pay.

**Examples**

- `hours = 45, rate = 10` → `475.0`
- `hours = 30, rate = 20` → `600.0`
- `hours = 40, rate = 15` → `600.0`

In [ ]:
def weekly_pay(hours, rate):
    """
    Pay is rate per hour for the first 40 hours and 1.5 × rate for every hour
    above 40. Return total pay.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(weekly_pay, [
    ((45, 10), 475.0),
    ((30, 20), 600.0),
    ((40, 15), 600.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Use if/else to handle the two cases.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Split hours with min and max: normal = min(hours, 40), extra = max(hours − 40, 0).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Use if/else to handle the two cases.

```python
def weekly_pay(hours, rate):
    if hours <= 40:
        pay = hours * rate
    else:
        pay = 40 * rate
        extra = hours - 40
        pay = pay + extra * rate * 1.5
    return float(pay)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Split hours with min and max: normal = min(hours, 40), extra = max(hours − 40, 0).

```python
def weekly_pay(hours, rate):
    return min(hours, 40) * rate + max(hours - 40, 0) * rate * 1.5
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Use if/else to handle the two cases. That costs O(1) time and O(1) space."

"I can do better: Split hours with min and max: normal = min(hours, 40), extra = max(hours − 40, 0). That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/operators/weekly-pay-with-overtime)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is the difference between / and //?</b></summary>

`/` is true division and always returns a float (7 / 2 = 3.5). `//` is floor division: it rounds the result down toward negative infinity (7 // 2 = 3, −7 // 2 = −4).

</details>

<details>
<summary><b>2. What does the % operator return for negative numbers?</b></summary>

In Python the result takes the sign of the divisor, so −7 % 3 = 2 and 7 % −3 = −2. This keeps `a == (a // b) * b + a % b` true.

</details>

<details>
<summary><b>3. What is short-circuit evaluation?</b></summary>

`and` and `or` evaluate left to right and stop as soon as the result is known: `False and …` never evaluates the right side; `True or …` neither.

</details>

<details>
<summary><b>4. What does x or y return?</b></summary>

Not necessarily a bool: it returns x if x is truthy, otherwise y. That's why `name or 'Guest'` gives a default.

</details>

<details>
<summary><b>5. Does Python have ++ and --?</b></summary>

No. Use `x += 1` and `x -= 1`. `++x` is valid syntax but just means +(+x), which doesn't change x.

</details>

<details>
<summary><b>6. What is 2 \*\* 3 \*\* 2?</b></summary>

512. The power operator is right-associative, so it's 2 \*\* (3 \*\* 2) = 2 \*\* 9.

</details>

<details>
<summary><b>7. What is the walrus operator?</b></summary>

`:=`, the assignment expression (Python 3.8+). It assigns a value and returns it in one expression, e.g. `while (line := f.readline()):`.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Arithmetic operators
- [ ] I can explain: Comparison and logical operators
- [ ] I can explain: Assignment operators
- [ ] I can explain: Order of operations (precedence)
- [ ] I can explain: Short-circuit evaluation, identity and membership
- [ ] I can explain: Bitwise operators

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q17](#q17) Reverse a Number
- [ ] [Q18](#q18) Power of Two
- [ ] [Q19](#q19) Round to Nearest Multiple
- [ ] [Q20](#q20) Weekly Pay With Overtime

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.